In [2]:
from dotenv import load_dotenv
import requests

load_dotenv()

api_key = os.getenv("football_data_api_key")

In [3]:
import requests, json, time, os

url = "https://api.football-data.org/v4/competitions/PL/matches"
headers = {"X-Auth-Token": api_key}

os.makedirs("data/raw", exist_ok=True)

for season in [2023, 2024, 2025]:
    path = f"data/raw/matches_{season}.json"
    if os.path.exists(path):          # past seasons never change, skip refetch
        continue
    r = requests.get(url, headers=headers, params={"season": season})
    print(season, r.status_code)
    r.raise_for_status()
    with open(path, "w") as f:
        json.dump(r.json(), f)
    time.sleep(7)                     # stay under 10 calls/min

2023 200
2024 200
2025 200


In [5]:
import duckdb

con = duckdb.connect()

con.sql("""
    CREATE TABLE matches_clean AS
    SELECT
        m.season.id AS season_id,
        m.matchday AS match_day,
        m.id AS match_id,
        m.utcDate AS date,
        m.homeTeam.name AS home_team,
        m.awayTeam.name AS away_team,
        m.score.winner AS winner
    FROM (
        SELECT unnest(matches) AS m
        FROM read_json_auto('data/raw/matches_2023.json')
    )

""")

con.sql("SELECT * FROM matches_clean").show()

┌───────────┬───────────┬──────────┬─────────────────────┬───────────────────────────┬────────────────────────────┬───────────┐
│ season_id │ match_day │ match_id │        date         │         home_team         │         away_team          │  winner   │
│   int64   │   int64   │  int64   │      timestamp      │          varchar          │          varchar           │  varchar  │
├───────────┼───────────┼──────────┼─────────────────────┼───────────────────────────┼────────────────────────────┼───────────┤
│      1564 │         1 │   435943 │ 2023-08-11 19:00:00 │ Burnley FC                │ Manchester City FC         │ AWAY_TEAM │
│      1564 │         1 │   435944 │ 2023-08-12 12:00:00 │ Arsenal FC                │ Nottingham Forest FC       │ HOME_TEAM │
│      1564 │         1 │   435945 │ 2023-08-12 14:00:00 │ AFC Bournemouth           │ West Ham United FC         │ DRAW      │
│      1564 │         1 │   435946 │ 2023-08-12 14:00:00 │ Brighton & Hove Albion FC │ Luton Town FC    